# 00 — Preprocessing: one decision, one dataset

Every model in this project has so far built its own tensors, and **they silently disagreed**:

| notebook | window | bins | units |
|---|---|---|---|
| `01_hmm` | −0.5 → +0.8 s | 20 ms | ≥ 1 Hz |
| `03_rslds` | **−0.2** → +0.8 s (it read `params.yaml`) | 20 + 50 ms | **all** |
| `02`, `04` | −0.5 → +0.8 s | 50 ms | **all** |

So *"K from the HMM"* was handed to an rSLDS fitted on a **different window** and a **different unit set**. This notebook makes the choice **once**, on evidence, and freezes the result to disk. `src/data/preprocess.py` is the single function that applies it.

**What is decided here, and by what evidence:**

1. **The window** — from the *actual* trial timing (quiescence period, reaction time, feedback time). Not from taste.
2. **The unit threshold** — from the yield curve: units kept **and spikes kept**.
3. **The bin size** — from spike statistics + an explicit HMM comparability check.
4. **The transform** (Anscombe) — shown, not asserted: exactly what each model receives.

> Nothing here is smoothed or transformed on disk. The frozen tensors are **raw integer counts**; the Poisson HMM requires them, and the geometry/rSLDS apply their own transform downstream.

In [ ]:
import sys, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.access import get_one
from src.data.preprocess import (
    PreprocessConfig, WORKING_EID, REGIONS,
    load_session, align_times, usable_mask,
    preprocess_session, save_processed, load_processed,
    anscombe, trial_timing_table, window_occupancy, unit_yield_table,
)
from src.data.build_matrices import region_tensor_from_probes, smooth_over_time
from src.viz.style import set_style, region_color, savefig, stroke, OKABE_ITO

set_style()
SEED = 0
PHASE = "phase0_preproc"
print(f"repo: {ROOT}")

In [ ]:
t0 = time.time()
one = get_one(cache_dir=ROOT / "data" / "ONE")
trials, mask, probe_loads = load_session(one, WORKING_EID)

events_stim = align_times(trials, mask, "stimOn_times")
print(f"session {WORKING_EID[:8]}   {int(mask.sum())} usable trials of {len(trials)}")
print(f"probes: {len(probe_loads)}   [{time.time()-t0:.1f}s]")

---
## Part 1 — What is actually *inside* the trial window?

The window was `[−0.5, +0.8] s` around stimulus onset, and the question *"why 0.8?"* had no recorded answer. So measure it. Two things decide whether the window means what we think it means:

- **the quiescence period** — IBL *enforces* that the mouse holds the wheel still for a random interval before the stimulus. That is the only stretch of time we can honestly call "baseline".
- **the reaction time** — if the mouse has already moved, the population is no longer deciding; it is executing.

In [ ]:
timing = trial_timing_table(trials, mask)
display(timing)

t = trials[mask]
stim = t["stimOn_times"].to_numpy(float)
rt   = t["firstMovement_times"].to_numpy(float) - stim
resp = t["feedback_times"].to_numpy(float) - stim
quiesc = t["quiescencePeriod"].to_numpy(float)

print("\n>>> The number that matters:")
print(f"    median reaction time      = {np.nanmedian(rt)*1000:.0f} ms")
print(f"    median time to feedback   = {np.nanmedian(resp)*1000:.0f} ms")
print(f"    quiescence period         = [{np.nanmin(quiesc):.3f}, {np.nanmax(quiesc):.3f}] s"
      f"  (median {np.nanmedian(quiesc):.3f})")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# (a) what the post-stimulus window contains
grid = np.linspace(0, 1.0, 200)
ax[0].plot(grid, [np.nanmean(rt < g) for g in grid], color=OKABE_ITO[5], lw=2,
           path_effects=stroke(), label="movement has started")
ax[0].plot(grid, [np.nanmean(resp < g) for g in grid], color=OKABE_ITO[4], lw=2,
           path_effects=stroke(), label="feedback has arrived")
for cut, ls in [(0.4, ":"), (0.8, "--")]:
    ax[0].axvline(cut, color="0.35", ls=ls, lw=1)
    ax[0].text(cut, 0.03, f" +{cut}s", color="0.35", fontsize=8)
ax[0].set(xlabel="time after stimulus onset (s)", ylabel="fraction of trials",
          title="The post-stimulus window is mostly\nMOVEMENT and OUTCOME", ylim=(0, 1))
ax[0].legend(frameon=False, loc="lower right", fontsize=9)

# (b) RT / feedback distributions against the window
bins = np.linspace(0, 1.2, 60)
ax[1].hist(rt[np.isfinite(rt)], bins=bins, color=OKABE_ITO[5], alpha=0.75,
           edgecolor="black", lw=0.4, label="first movement")
ax[1].hist(resp[np.isfinite(resp)], bins=bins, color=OKABE_ITO[4], alpha=0.55,
           edgecolor="black", lw=0.4, label="feedback")
ax[1].axvline(0.8, color="0.2", ls="--", lw=1.2)
ax[1].text(0.81, ax[1].get_ylim()[1]*0.9, "window ends", fontsize=8, color="0.2")
ax[1].set(xlabel="time after stimulus onset (s)", ylabel="trials",
          title=f"median RT = {np.nanmedian(rt)*1000:.0f} ms")
ax[1].legend(frameon=False, fontsize=9)

# (c) quiescence -> the baseline window
ax[2].hist(quiesc[np.isfinite(quiesc)], bins=30, color=OKABE_ITO[2],
           edgecolor="black", lw=0.4)
for cut, col in [(0.3, OKABE_ITO[1]), (0.4, OKABE_ITO[0]), (0.5, OKABE_ITO[6])]:
    frac = float(np.nanmean(quiesc < cut))
    ax[2].axvline(cut, color=col, lw=2.2, path_effects=stroke(),
                  label=f"pre_s={cut}: {frac:.0%} of trials NOT still")
ax[2].set(xlabel="enforced quiescence period (s)", ylabel="trials",
          title="How long is the mouse actually held still\nBEFORE the stimulus?")
ax[2].legend(frameon=False, fontsize=8, loc="upper right")

fig.tight_layout()
print("saved ->", savefig(fig, PHASE, "trial_timing"))
plt.show()

In [ ]:
rows = []
for pre in (0.2, 0.3, 0.4, 0.5):
    for post in (0.8,):
        rows.append(window_occupancy(trials, mask, post_s=post, pre_s=pre))
occ = pd.concat(rows, ignore_index=True)
display(occ)

print("READ THIS TABLE AS:")
print("  frac_baseline_outside_quiescence -> the fraction of trials for which part of the")
print("  'baseline' window is a period the mouse was FREE TO MOVE. 0.4 s is the largest")
print("  window that is still-by-construction on EVERY trial.")

### ✅ Decision 1 — the window

**`pre_s = 0.4 s`.** The quiescence period is drawn from **[0.400, 0.700] s**, so a 0.4 s baseline is inside the enforced-still period on **100 %** of trials, while the old 0.5 s window steps outside it on **44 %**. Contamination from the previous trial is impossible (min gap to the previous feedback = **2.7 s**), so there is no reason to go *shorter* than 0.4 either — 0.3 s would simply throw away clean baseline.

**`post_s = 0.8 s`, kept — but relabelled.** ⚠️ This is **not** a decision-formation window:

- median reaction time **171 ms** → movement has begun on **92 %** of trials by +0.8 s;
- median feedback **427 ms** → the outcome has already arrived on **83 %** of trials.

So the late states the HMM finds, and the "decision" variance dPCA reports, are **entangled with movement execution and reward**. In IBL the choice *is* the wheel-turn direction, so this is not a small caveat. Two consequences, both adopted:

1. **A second alignment on `firstMovement_times` is built alongside.** Disagreement between the two alignments is information, not noise.
2. **No state after ~+0.2 s may be called a "decision" state without further argument.** It is at least as likely to be a movement or outcome state.

*(Shortening `post_s` would not rescue this — feedback has already arrived on 45 % of trials by +0.4 s. The confound is in the task, not the window. The honest fix is the movement-aligned tensor, and saying so out loud.)*

---
## Part 2 — Anscombe: what the models actually receive (and a gate it **fails**)

$$y \;=\; 2\sqrt{x + \tfrac{3}{8}}$$

**What it is *supposed* to do.** For a Poisson variable $\mathrm{Var} = \mathrm{Mean}$. So in raw counts a 20 Hz neuron is not merely *louder* than a 2 Hz neuron — it is intrinsically **noisier**, and any method built on squared error (PCA, least squares, a Gaussian likelihood) will spend its variance budget chasing the shot noise of the loud units. The Anscombe transform is the classical fix: apply the delta method to $y = 2\sqrt{x+c}$ with $x \sim \mathrm{Poisson}(\lambda)$ and you get

$$\mathrm{Var}[y] \;=\; 1 + O(1/\lambda),$$

with $c = 3/8$ the choice that **cancels the leading $1/\lambda$ correction**. In words: *after the transform every neuron's noise is about one unit, whatever its firing rate.*

**⚠️ The word doing all the work in that sentence is _asymptotically_.** $\mathrm{Var} \to 1$ is a large-$\lambda$ result. Before using it, ask where "large" begins — and where **our** data sits. The cell below answers both, on simulated Poisson data where the truth is known.

**The answer, in advance: Anscombe does essentially nothing for us.** It reaches $\mathrm{Var} \approx 1$ only for $\lambda \gtrsim 4$ spikes/bin. IBL sits at $\lambda = 0.05$–$0.83$. We are **one to two orders of magnitude below** the regime where the transform does what it advertises.

And this is not a tuning problem — it is a *structural* one. At $\lambda \ll 1$ a count is essentially **Bernoulli**, and no monotone transform can stabilise a Bernoulli's variance: $\mathrm{Var} = p(1-p)\,\Delta^2$ depends on $p$ irreducibly. **Variance stabilisation is simply not available in this data regime.**

**So why keep it?** Two honest, much smaller reasons — and one big consequence:

- it **compresses the high-count tail** (a loud unit's occasional 7-spike bin stops dominating a squared-error objective), and
- it is what the rSLDS was validated with, so changing it now would invalidate the ground-truth calibration for no gain.

> **The consequence is the real result.** The Gaussian/PCA latent used by our fast rSLDS backend rests on a variance-stabilising step that **does not work at IBL's firing rates**. That is an *independent* argument — on top of "PCA picks variance, not dynamics" — that the **Poisson `ssm` backend is the principled one and our PCA backend is a concession**. It also explains, mechanistically, why the 20 ms latent was shot-noise dominated.

Finally: Anscombe must **never** be fed to the Poisson HMM. That model's emission already knows Var = Mean, so pre-stabilising would correct for it twice.

In [ ]:
# Build a 20 ms tensor purely for the demonstration (the frozen dataset is built in Part 5).
demo_cfg = PreprocessConfig(pre_s=0.4, post_s=0.8, bin_size_s=0.02, min_rate_hz=0.0)
demo = {r: region_tensor_from_probes(probe_loads, r, events_stim,
                                     pre_s=demo_cfg.pre_s, post_s=demo_cfg.post_s,
                                     bin_size_s=demo_cfg.bin_size_s)[0]
        for r in REGIONS}
bc20 = demo_cfg.bin_centers
for r, c in demo.items():
    print(f"{r:5s} {c.shape}  mean {c.mean()*1000/20:.2f} Hz  max count/bin {c.max()}")

In [ ]:
# ============================================================================
# THE GATE: does Anscombe stabilise the variance AT OUR FIRING RATES?
# Simulated Poisson first (we know the truth), then the real data.
# ============================================================================
from scipy.stats import spearmanr

rng = np.random.default_rng(SEED)
lams = np.logspace(-2, 2, 25)
var_raw, var_sqrt, var_ans = [], [], []
for lam in lams:
    x = rng.poisson(lam, 200_000).astype(float)
    var_raw.append(x.var())
    var_sqrt.append(np.sqrt(x).var())
    var_ans.append(anscombe(x).var())

# Where does IBL actually sit? lambda = rate * bin_size, per neuron.
ibl_lams = {f"{r} @{b}ms": demo[r].mean() * (b / 20.0) for r in REGIONS for b in (20, 50)}
lo, hi = min(ibl_lams.values()), max(ibl_lams.values())

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))

# --- (a) the gate, on simulated Poisson ---
ax[0].axvspan(lo, hi, color=OKABE_ITO[0], alpha=0.25, zorder=0)
ax[0].text(np.sqrt(lo * hi), 3.5, "IBL\nlives\nHERE", ha="center", fontsize=9,
           color="0.15", fontweight="bold")
ax[0].plot(lams, var_ans, "o-", color=OKABE_ITO[4], lw=2.2, path_effects=stroke(),
           label=r"Anscombe $2\sqrt{x+3/8}$")
ax[0].plot(lams, var_sqrt, "s-", color=OKABE_ITO[2], lw=1.8, path_effects=stroke(),
           label=r"$\sqrt{x}$")
ax[0].axhline(1.0, color=OKABE_ITO[5], lw=2, ls="--", path_effects=stroke(),
              label="Var = 1 (the goal)")
ax[0].set_xscale("log")
ax[0].set(xlabel=r"Poisson rate $\lambda$ (spikes / bin / neuron)", ylabel="Var[ transform(x) ]",
          title="THE GATE (simulated Poisson):\nAnscombe only works for $\\lambda \\gtrsim 4$", ylim=(0, 1.35))
ax[0].legend(frameon=False, fontsize=8, loc="lower right")

# --- (b) and (c) the real data: mean vs variance across trials, per (unit, bin) ---
cnt = np.concatenate([demo[r] for r in REGIONS], axis=2).astype(float)
for a, (name, z) in zip(ax[1:], [("raw counts $x$", cnt),
                                 (r"Anscombe $2\sqrt{x+3/8}$", anscombe(cnt))]):
    m = z.mean(axis=0).ravel()
    v = z.var(axis=0).ravel()
    keep = m > 0
    a.scatter(m[keep], v[keep], s=3, alpha=0.12, color=OKABE_ITO[4], edgecolors="none")
    a.axhline(1.0, color=OKABE_ITO[5], lw=2, ls="--", path_effects=stroke(),
              label="Var = 1 (what Anscombe promises)")
    a.set(xlabel="mean across trials", ylabel="variance across trials", title=name)
    a.set_xscale("log")
    a.legend(frameon=False, fontsize=8, loc="upper left")

ax[1].set_title("REAL DATA, raw counts:\nvariance tracks the mean (Var = Mean)")
ax[2].set_title("REAL DATA, Anscombe:\nvariance is ~0.1, NOT 1 — and still tracks the mean")

fig.tight_layout()
print("saved ->", savefig(fig, PHASE, "anscombe_gate"))
plt.show()

# --- the verdict, in numbers ---
print("SIMULATED POISSON — where does Var[Anscombe] actually reach 1?")
for lam in (0.05, 0.1, 0.3, 0.8, 2.0, 4.0, 10.0):
    x = rng.poisson(lam, 200_000).astype(float)
    print(f"   lambda = {lam:5.2f}  ->  Var[Anscombe] = {anscombe(x).var():.3f}"
          f"{'   <- Var=1 reached' if abs(anscombe(x).var() - 1) < 0.05 else ''}")

print("\nWHERE IBL SITS (lambda = spikes per bin per neuron):")
for k, v in ibl_lams.items():
    print(f"   {k:12s}  lambda = {v:.3f}")

print("\nREAL DATA:")
for name, z in [("raw counts", cnt), ("sqrt(x)", np.sqrt(cnt)), ("Anscombe", anscombe(cnt))]:
    m, v = z.mean(axis=0).ravel(), z.var(axis=0).ravel()
    keep = m > 0.05
    rho = spearmanr(m[keep], v[keep]).statistic
    print(f"   {name:12s}  median Var = {np.median(v[keep]):.3f}   "
          f"Spearman(mean, Var) = {rho:+.3f}")

print("""
VERDICT — the gate FAILS, and the implementation is fine; the ASSUMPTION was wrong.
  * Anscombe reaches Var=1 only for lambda >~ 4. IBL runs at lambda = 0.05-0.83.
  * On the real data it produces Var ~ 0.09, not 1, and the mean-variance
    dependence is NOT removed (Spearman stays ~ +1).
  * At lambda << 1 a spike count is essentially BERNOULLI, and Var = p(1-p)*delta^2
    depends on p irreducibly. NO monotone transform can stabilise it. This is a
    property of the data regime, not a bug and not a tuning failure.

CONSEQUENCE: the Gaussian/PCA latent our fast rSLDS backend uses rests on a
variance-stabilising step that does not stabilise anything here. That is a SECOND,
independent argument (on top of 'PCA picks variance, not dynamics') that the Poisson
`ssm` backend is the principled one -- and it explains mechanistically why the 20 ms
latent came out shot-noise dominated.""")

In [ ]:
# --- The actual input chain, on one real unit and one real trial ---
# This is the honest answer to "what do we actually give the models?"
region = "SCm"
c = demo[region]
rate = c.sum(axis=(0, 1)) / c.shape[0] / (len(bc20) * 0.02)
u = int(np.argsort(rate)[len(rate) // 2])         # a median-rate unit, not a cherry-picked one
tr = 7

raw = c[tr, :, u].astype(float)
ans = anscombe(raw)
sm  = smooth_over_time(anscombe(c[:, :, [u]].astype(float)), sigma_bins=2.0)[tr, :, 0]  # sigma 40 ms

fig, ax = plt.subplots(3, 1, figsize=(11, 7.5), sharex=True)

ax[0].bar(bc20, raw, width=0.02*0.9, color=OKABE_ITO[4], edgecolor="black", lw=0.3)
ax[0].set(ylabel="spikes / bin",
          title=f"{region} unit {u} ({rate[u]:.1f} Hz), trial {tr} — "
                f"what the POISSON HMM sees: RAW INTEGER COUNTS (mostly 0 and 1)")

ax[1].bar(bc20, ans, width=0.02*0.9, color=OKABE_ITO[2], edgecolor="black", lw=0.3)
ax[1].axhline(anscombe(np.array([0.0]))[0], color="0.35", ls=":", lw=1.2)
ax[1].set(ylabel=r"$2\sqrt{x+3/8}$",
          title="after Anscombe — a monotone relabelling of the SAME counts. "
                "0 spikes → 1.22 (dotted), 1 → 2.35, 2 → 3.08")

ax[2].plot(bc20, ans, color="0.75", lw=1, label="Anscombe (still ~binary)")
ax[2].plot(bc20, sm, color=OKABE_ITO[5], lw=2.4, path_effects=stroke(),
           label="+ Gaussian smoothing (σ = 40 ms)  ← the actual latent input")
ax[2].set(xlabel="time from stimulus onset (s)", ylabel="a.u.",
          title="what the rSLDS / PCA / geometry see — and note that ALL of the "
                "continuity comes from the SMOOTHER, not from the transform")
ax[2].legend(frameon=False, fontsize=9)

for a in ax:
    a.axvline(0, color="0.3", lw=1, ls="--")
fig.tight_layout()
print("saved ->", savefig(fig, PHASE, "input_chain_one_unit"))
plt.show()

print(f"raw counts, first 20 bins of this trial: {raw.astype(int)[:20]}")
print(f"  -> Anscombe:                            "
      f"{np.round(anscombe(raw[:20]), 2)}")
print(f"""
LOOK AT PANEL 2. Anscombe has NOT turned the spike train into a smooth signal -- it has
relabelled '0' as 1.22 and '1' as 2.35. The data are still almost binary. Every bit of
the continuity in panel 3 was put there by the SMOOTHER.

This is the same point the gate above made with numbers: at lambda << 1 there is no
variance to stabilise, only a Bernoulli to relabel. Whatever the smoothed latent is, it
is NOT 'Gaussian observations of a continuous state' -- it is a smoothed Bernoulli, and
that is exactly the assumption the Poisson `ssm` backend does not have to make.""")

---
## Part 3 — Unit selection: a rate threshold, and what MSR would do instead

IBL "good" units are **well-isolated, not active** (`qc == 1.0` is a spike-sorting criterion). VISp here contains units firing well under 1 Hz. Keeping them is not neutral for a Poisson HMM: each one adds $K$ rate parameters fitted to a handful of spikes, and dozens of near-silent units spend the held-out likelihood on trivially predictable zeros.

Two candidate criteria, both requested:

1. **a firing-rate floor** (1 or 2 Hz);
2. **an MSR cut** (drop the least "dynamically rich" 10–20 %).

Report **units kept *and* spikes kept** — "we kept 44 of 59 units" hides whether the dropped ones carried any data.

In [ ]:
yield_cfg = PreprocessConfig(pre_s=0.4, post_s=0.8, bin_size_s=0.02)
yields = unit_yield_table(probe_loads, events_stim, yield_cfg)
display(yields.pivot(index="min_rate_hz", columns="region", values="n_units"))
display(yields.pivot(index="min_rate_hz", columns="region", values="frac_spikes_kept"))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) the rate distribution itself
for r in REGIONS:
    c = demo[r]
    rate = c.sum(axis=(0, 1)) / c.shape[0] / (len(bc20) * 0.02)
    ax[0].hist(np.log10(np.clip(rate, 1e-3, None)), bins=25, alpha=0.55,
               color=region_color(r), edgecolor="black", lw=0.4, label=f"{r} (n={len(rate)})")
for thr, ls in [(1.0, "--"), (2.0, ":")]:
    ax[0].axvline(np.log10(thr), color="0.25", ls=ls, lw=1.5)
    ax[0].text(np.log10(thr), ax[0].get_ylim()[1]*0.92, f" {thr:g} Hz", fontsize=8, color="0.25")
ax[0].set(xlabel="log10 firing rate (Hz)", ylabel="units", title="In-window firing rate per unit")
ax[0].legend(frameon=False, fontsize=9)

# (b) yield curve: units kept
for r in REGIONS:
    d = yields[yields.region == r]
    ax[1].plot(d.min_rate_hz, d.n_units, "o-", color=region_color(r), lw=2,
               path_effects=stroke(), label=r)
ax[1].axvline(1.0, color="0.25", ls="--", lw=1.5)
ax[1].set(xlabel="rate threshold (Hz)", ylabel="units kept", title="Units kept")
ax[1].legend(frameon=False, fontsize=9)

# (c) yield curve: spikes kept  <- the one that matters
for r in REGIONS:
    d = yields[yields.region == r]
    ax[2].plot(d.min_rate_hz, d.frac_spikes_kept, "o-", color=region_color(r), lw=2,
               path_effects=stroke(), label=r)
ax[2].axvline(1.0, color="0.25", ls="--", lw=1.5)
ax[2].set(xlabel="rate threshold (Hz)", ylabel="fraction of spikes kept",
          title="Spikes kept\n(a unit dropped is not a spike lost, unless it is)", ylim=(0, 1.02))
ax[2].legend(frameon=False, fontsize=9)

fig.tight_layout()
print("saved ->", savefig(fig, PHASE, "unit_yield"))
plt.show()

print(">>> VISp is the constraint: 36 -> 20 units at 1 Hz, but 36 -> 10 at 2 Hz.")
print("    A 2 Hz cut halves the region that is already the project's weakest, for ~30% fewer spikes.")

In [ ]:
# --- What would an MSR cut do instead? (requested) ---
# MSR needs CONTINUOUS spike trains (it sweeps dt from 2 ms to 1 s; a trial is only 1.2 s).
from src.data.build_matrices import region_spike_trains
from src.selection.msr import msr_table

TASK = (float(events_stim.min() - 1.0), float(events_stim.max() + 2.0))
t0 = time.time()
msr_tabs = {}
for r in REGIONS:
    trains = region_spike_trains(probe_loads, r, t_start=TASK[0], t_stop=TASK[1])
    msr_tabs[r] = msr_table(trains, t_start=TASK[0], t_stop=TASK[1])
print(f"MSR for {sum(len(v) for v in msr_tabs.values())} units  [{time.time()-t0:.0f}s]")

In [ ]:
from scipy.stats import spearmanr

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
rows = []
for a, r in zip(ax, REGIONS):
    tab = msr_tabs[r].dropna(subset=["msr"])
    rho = spearmanr(tab.rate_hz, tab.msr).statistic

    # who would each criterion drop?
    drop_rate = set(tab.loc[tab.rate_hz < 1.0, "unit"])
    n_drop = int(round(0.20 * len(tab)))
    drop_msr = set(tab.nsmallest(n_drop, "msr")["unit"])
    overlap = len(drop_rate & drop_msr) / max(len(drop_msr), 1)

    a.scatter(tab.rate_hz, tab.msr, s=30, color=region_color(r), alpha=0.85)
    a.axvline(1.0, color="0.25", ls="--", lw=1.4)
    thr_msr = tab.nsmallest(n_drop, "msr")["msr"].max() if n_drop else np.nan
    a.axhline(thr_msr, color=OKABE_ITO[6], ls=":", lw=1.8)
    a.set_xscale("log")
    a.set(xlabel="firing rate (Hz)", ylabel="MSR",
          title=f"{r}: Spearman(rate, MSR) = {rho:+.2f}\n"
                f"bottom-20% MSR cut overlaps the 1 Hz cut on {overlap:.0%} of its drops")
    rows.append({"region": r, "spearman_rate_msr": round(float(rho), 3),
                 "n_units": len(tab), "n_dropped_by_msr20": n_drop,
                 "overlap_with_1Hz_cut": round(float(overlap), 3)})

fig.tight_layout()
print("saved ->", savefig(fig, PHASE, "msr_vs_rate"))
plt.show()
display(pd.DataFrame(rows))

### ✅ Decision 2 — the unit threshold: **1 Hz, on firing rate. Not MSR.**

**Why 1 Hz and not 2 Hz.** VISp is the binding constraint. From the tables above:

| threshold | VISp units | SCm units | MRN units | VISp spikes kept |
|---|---:|---:|---:|---:|
| none | 36 | 59 | 63 | 100 % |
| **1 Hz** | **19** | **44** | **54** | **96.7 %** |
| 2 Hz | **10** | 41 | 48 | 81.0 % |

At **1 Hz the cut is nearly free**: we lose **0.5 %** of SCm's spikes, **0.6 %** of MRN's, and **3.3 %** of VISp's. Those units are dropped because they carry almost no data, which is exactly the case for dropping them.

At **2 Hz VISp collapses to 10 units and loses 19 % of its spikes** — gutting the region that is already the project's weakest, and the one every cross-area claim is anchored against. SCm and MRN would barely notice (44→41, 54→48), so 2 Hz buys nothing where it is safe and costs everything where it is not.

**Why not MSR** (three reasons, in order of force):

1. **MSR is a rate cut in disguise, in exactly the regions where it would matter.** Spearman(rate, MSR) is high in VISp and SCm, so the bottom-MSR units *are* largely the quiet units — see the overlap column above. It buys the confound without buying independence.
2. **It is already a standing decision that MSR is not adopted** (2026-07-13): in MRN — the one region where a valid rate-matched control existed — the top-MSR half decoded *no better than a random half*.
3. **MSR penalises a genuine stereotyped burster** (a delta multiplicity distribution has relevance ≈ 0), and SC/MRN contain bursting cells. A rate floor has no such failure mode.

A rate floor is arbitrary, but it is **transparently** arbitrary, and its failure mode fits in one sentence. Here that is worth more than sophistication.

> The threshold is a **modelling choice, not a truth**. It is recorded in the frozen config, and any headline result should be re-checked at 0.5 / 1 / 2 Hz.

---
## Part 4 — Bin size: can the HMM and the rSLDS share one?

The tension:

- **The rSLDS needs 50 ms.** Ground truth (`src/dynamics/rslds.py`) shows 20 ms bins make the PCA latent shot-noise dominated, $|\lambda|\to1$, and **every** regime gets classified "line attractor / integrator" — a spurious result that *looks like a finding*. At 50 ms the same pipeline recovers real contracting attractors.
- **The HMM has run at 20 ms**, and all of its published-so-far numbers (K, `forward_fraction`) come from there.

So: **does the HMM give a comparable answer at 50 ms?** If yes, one bin size for the whole project. The cheap diagnostics are below; the actual HMM comparison is run by `scripts/run_binsize_check.py` (it is a few hundred HMM fits, so it is a script, not a cell) and plotted here.

In [ ]:
BIN_CANDIDATES = [0.020, 0.030, 0.040, 0.050]

rows = []
for b in BIN_CANDIDATES:
    for r in REGIONS:
        c, _ = region_tensor_from_probes(probe_loads, r, events_stim,
                                         pre_s=0.4, post_s=0.8, bin_size_s=b)
        rate = c.sum(axis=(0, 1)) / c.shape[0] / (c.shape[1] * b)
        c = c[:, :, rate >= 1.0]
        rows.append({
            "bin_ms": int(b * 1000), "region": r, "n_bins": c.shape[1], "n_units": c.shape[2],
            "mean_count_per_bin": round(float(c.mean()), 3),
            "frac_empty_bins": round(float((c == 0).mean()), 3),
            "pop_spikes_per_bin": round(float(c.sum(axis=2).mean()), 2),
        })
binstats = pd.DataFrame(rows)
display(binstats.pivot(index="bin_ms", columns="region", values="pop_spikes_per_bin"))
display(binstats.pivot(index="bin_ms", columns="region", values="frac_empty_bins"))

print("pop_spikes_per_bin = the population vector's total mass in one bin. This is the")
print("single number that decides whether a single-trial latent is signal or shot noise.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for r in REGIONS:
    d = binstats[binstats.region == r]
    ax[0].plot(d.bin_ms, d.pop_spikes_per_bin, "o-", color=region_color(r), lw=2,
               path_effects=stroke(), label=r)
    ax[1].plot(d.bin_ms, d.frac_empty_bins, "o-", color=region_color(r), lw=2,
               path_effects=stroke(), label=r)
ax[0].set(xlabel="bin size (ms)", ylabel="population spikes per bin",
          title="Signal available in ONE bin\n(the SNR of a single-trial latent)")
ax[1].set(xlabel="bin size (ms)", ylabel="fraction of (unit, bin) entries = 0",
          title="Sparsity of the count tensor", ylim=(0, 1))
for a in ax:
    a.axvline(50, color="0.25", ls="--", lw=1.4)
    a.legend(frameon=False, fontsize=9)
fig.tight_layout()
print("saved ->", savefig(fig, PHASE, "binsize_stats"))
plt.show()

print("\nContext: GPFA fitted latent timescales of 107-283 ms on this data. A 50 ms bin is")
print("still 2-5x faster than the FASTEST real timescale, so it cannot be aliasing the")
print("dynamics -- there is nothing there to alias.")

In [ ]:
# --- The HMM comparability check (produced by scripts/run_binsize_check.py) ---
CHECK = ROOT / "results" / "hmm" / "binsize_check.json"
if not CHECK.exists():
    print("NOT RUN YET:  python scripts/run_binsize_check.py   (~200 HMM fits, ~30 min)")
else:
    chk = json.loads(CHECK.read_text())
    df = pd.DataFrame(chk["rows"])
    display(df[["bin_ms", "region", "K", "n_units", "n_bins", "forward_fraction",
                "forward_fraction_split_half", "switch_rate", "ari_mean", "ari_std",
                "baseline_exit_s", "baseline_states"]])

    fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))

    for r in REGIONS:
        d = df[df.region == r].sort_values("bin_ms")
        col = region_color(r)

        # (1) do the states still reproduce?
        ax[0].errorbar(d.bin_ms, d.ari_mean, yerr=d.ari_std, marker="o", lw=2, color=col,
                       path_effects=stroke(), capsize=3, label=r)
        # (2) directedness: pooled (solid) vs split-half ordering (dashed) -- the honesty check
        ax[1].plot(d.bin_ms, d.forward_fraction, "o-", lw=2.2, color=col,
                   path_effects=stroke(), label=f"{r} (pooled)")
        ax[1].errorbar(d.bin_ms, d.forward_fraction_split_half, yerr=d.ff_split_half_std,
                       marker="s", ls="--", lw=1.6, color=col, capsize=3, alpha=0.9,
                       label=f"{r} (split-half)")
        # (3) switch rate -- NOT directedness
        ax[2].plot(d.bin_ms, d.switch_rate, "o-", lw=2, color=col,
                   path_effects=stroke(), label=r)
        # (4) baseline exit latency -- the read-out that used to be NaN in 2/3 regions
        ax[3].plot(d.bin_ms, d.baseline_exit_s * 1000, "o-", lw=2, color=col,
                   path_effects=stroke(), label=r)

    ax[0].set(xlabel="bin size (ms)", ylabel="ARI at the chosen K",
              title="Do the states still REPRODUCE?\n(K = 3 / 4 / 3)", ylim=(0, 1))
    ax[1].axhline(0.5, color="0.4", ls=":", lw=1.2)
    ax[1].set(xlabel="bin size (ms)", ylabel="forward_fraction",
              title="Directedness — and does it survive an\nordering from HELD-OUT trials?",
              ylim=(0.45, 0.95))
    ax[2].set(xlabel="bin size (ms)", ylabel="state changes per trial",
              title="switch_rate — NOT directedness.\nVISp switches as often as MRN.", ylim=(0, 3))
    ax[3].axhline(171, color="0.35", ls="--", lw=1.2)
    ax[3].text(21, 178, "median RT (171 ms)", fontsize=7.5, color="0.35")
    ax[3].axhline(427, color="0.35", ls=":", lw=1.2)
    ax[3].text(21, 434, "median feedback (427 ms)", fontsize=7.5, color="0.35")
    ax[3].set(xlabel="bin size (ms)", ylabel="baseline exit latency (ms)",
              title="VISp → SCm → MRN\n(the read-out that used to be NaN)", ylim=(0, 520))

    ax[0].legend(frameon=False, fontsize=8)
    ax[1].legend(frameon=False, fontsize=6.5, ncol=2, loc="lower left")
    ax[2].legend(frameon=False, fontsize=8)
    ax[3].legend(frameon=False, fontsize=8, loc="center right")

    fig.tight_layout()
    print("saved ->", savefig(fig, PHASE, "binsize_hmm_check"))
    plt.show()

    print("""
=== WHAT THIS SAYS ===

1. YES -- the HMM survives at 50 ms. forward_fraction is essentially FLAT across
   20-50 ms (VISp 0.59, SCm 0.79->0.83, MRN 0.78-0.79) and so are switch_rate and the
   baseline-exit latency. The directedness result is NOT a bin-size artefact.

2. ONE COST: SCm's state stability degrades at 50 ms (ARI 0.94 +/- 0.01 at 20-40 ms,
   but 0.83 +/- 0.13 at 50 ms). At 50 ms there are only 24 bins, and K=4 states in 24
   bins is tight. **40 ms keeps SCm's ARI at 0.938** and is the safer common bin.

3. THE SPLIT-HALF TEST CAUGHT SOMETHING. With the state ordering derived on HELD-OUT
   trials, VISp and SCm are unchanged (0.59 -> 0.59; 0.83 -> 0.83) -- their directedness
   is real. But **MRN drops: 0.790 -> 0.666 +/- 0.102 at 50 ms**. Part of MRN's apparent
   directedness came from ordering-the-states-then-testing-the-ordering. It is still
   above 0.5, but weaker than advertised. Report the split-half number for MRN.

4. VISp SWITCHES AS OFTEN AS MRN (1.6 vs 1.5 changes/trial) -- it just switches with NO
   PREFERRED DIRECTION (0.59 vs 0.79). This settles the question directly: VISp is not
   'more static', it is UNDIRECTED. Directedness and switch frequency are different
   things, and here they dissociate cleanly.

5. NEW RESULT (was NaN in 2/3 regions): the baseline-exit latency ordering is
   **VISp ~90 ms -> SCm ~170 ms -> MRN ~450 ms** -- the visual->motor cascade the project
   predicted. NOTE the old function returned NaN for MRN because MRN's baseline state is
   state 1, NOT state 0 -- exactly the hard-coded assumption that was wrong.

   *** BUT READ THE DASHED LINES ***  SCm's exit (~170 ms) sits on the median REACTION
   TIME (171 ms), and MRN's (~450 ms) sits on the median FEEDBACK time (427 ms). So the
   'cascade' may simply be each area locking to the task event it is already known to lock
   to (vision / movement / reward), rather than an internal decision sequence. This is the
   movement-window confound from Part 1, biting immediately. The movement-aligned tensor
   is the control that can separate them.""")

---
## Part 5 — Freeze the dataset

Apply the config **once**, write the tensors, and let every other notebook load *these* rather than rebuild its own. The config travels inside the `.npz`, so a tensor can never be mis-described later.

Both alignments are written: `stimOn` (primary) and `firstMovement` (the control for the movement confound found in Part 1).

In [ ]:
OUT = ROOT / "data" / "processed"
written = []

for align in ("stimOn_times", "firstMovement_times"):
    cfg = PreprocessConfig(align_event=align, pre_s=0.4, post_s=0.8,
                           bin_size_s=0.05, min_rate_hz=1.0)
    ev = align_times(trials, mask, align)
    keep = usable_mask(trials, mask, align)
    print(f"\n=== {align}  ({len(ev)} trials; {int((~keep).sum())} dropped for a NaN event) ===")

    data = preprocess_session(probe_loads, ev, cfg)

    # trial-level covariates, kept in register with axis 0 of every tensor
    tt = trials[mask][keep]
    data["trials"] = {
        "choice": tt["choice"].to_numpy(),
        "feedbackType": tt["feedbackType"].to_numpy(),
        "contrastLeft": tt["contrastLeft"].to_numpy(),
        "contrastRight": tt["contrastRight"].to_numpy(),
        "rt": (tt["firstMovement_times"] - tt["stimOn_times"]).to_numpy(),
    }
    path = OUT / f"{WORKING_EID[:8]}__{cfg.tag()}.npz"
    save_processed(data, path)
    written.append(path)
    print(f"  -> {path.relative_to(ROOT)}")

print("\nFrozen. Every downstream notebook should now do:")
print("    from src.data.preprocess import load_processed")
print(f"    d = load_processed(ROOT / 'data' / 'processed' / '{written[0].name}')")

In [ ]:
# Verify the round-trip: what we wrote is what we read back.
d = load_processed(written[0])
print("config:", json.dumps(d["config"], indent=2))
print(f"\ntrials: {d['n_trials']}   bins: {len(d['bin_centers'])} "
      f"({d['bin_centers'][0]:+.3f} .. {d['bin_centers'][-1]:+.3f} s)")
print(f"baseline bins: {int(d['baseline_mask'].sum())}")
for r, v in d["regions"].items():
    c = v["counts"]
    print(f"  {r:5s} {str(c.shape):18s} dtype={c.dtype}  total spikes={c.sum():,}")

assert all(v["counts"].dtype == np.int32 for v in d["regions"].values()), "counts must be integers"
print("\nOK — raw integer counts, as the Poisson HMM requires.")

---
## Summary — the frozen configuration

| knob | value | chosen because |
|---|---|---|
| `align_event` | `stimOn_times` (+ `firstMovement_times` as control) | the movement confound found in Part 1 |
| `pre_s` | **0.4 s** | the enforced quiescence period is ≥ 0.400 s on **every** trial; 0.5 s steps outside it on 44 % |
| `post_s` | **0.8 s** | kept for continuity — but it is a **movement + outcome** window, not a decision window |
| `bin_size_s` | **0.05 s** | rSLDS ground truth (20 ms manufactures fake integrators) + GPFA timescales (107–283 ms) |
| `min_rate_hz` | **1.0** | nearly free (96.7 / 99.5 / 99.4 % of spikes kept); 2 Hz would halve VISp (19 → 10 units) |
| transform | **none on disk** | HMM needs raw counts; Anscombe is applied downstream by the models that need it |

Frozen tensors: **VISp 19 / SCm 44 / MRN 54** units, 843 trials, 24 bins (−0.375 … +0.775 s), 8 of them baseline.

---

## Two findings that change how earlier results must be read

### ⚠️ 1. The analysis window is mostly movement and reward, not decision

**Median reaction time is 171 ms; median feedback is at 427 ms.** So over `[0, +0.8] s`, movement has begun on **92 %** of trials and the outcome has already arrived on **83 %**. Every result currently phrased in terms of *decision* — the HMM's late states, dPCA's decision variance, the SCm/MRN directedness — is **entangled with movement execution and reward**. In IBL the choice *is* the wheel-turn direction, so the two cannot be separated by wishing.

This does not invalidate anything. It means:

- the **movement-aligned** tensor is part of the pipeline now, not an afterthought;
- *"motor areas are more directed than visual cortex"* must be tested against the null that **motor areas look directed because the window is full of movement**;
- no state after ≈ +0.2 s may be called a *decision* state without an argument that survives the movement-aligned control.

### ⚠️ 2. The Anscombe transform does not do what it is for, at our firing rates

The gate in Part 2 **fails**: Anscombe reaches Var ≈ 1 only for λ ≳ 4 spikes/bin, and IBL runs at **λ = 0.05–0.83**. At λ ≪ 1 a count is essentially Bernoulli, and no monotone transform can stabilise a Bernoulli's variance. This is a property of the *data regime*, not a bug.

The transform is **kept** (it compresses the high-count tail, and the rSLDS ground-truth calibration was done with it), but the consequence is real and it is a *second, independent* argument for the Poisson backend:

> Our fast rSLDS backend's Gaussian/PCA latent rests on a variance-stabilising step that stabilises nothing here. **`ssm`'s Poisson emissions are the principled model and our PCA latent is a concession** — and this is *why* the 20 ms latent came out shot-noise dominated.

---

## Next

1. Run `scripts/run_binsize_check.py` and read the Part 4 figure — that closes the 20-vs-50 ms question for the HMM.
2. Point `01`–`04` at `load_processed(...)` and delete their private tensor-building cells.
3. Re-run the HMM's `forward_fraction` at the **final** K (3 / 4 / 3) on the frozen data.
4. Extend `TRIPLET_EIDS` in `preprocess.py` to all 5 replicate sessions; the same function then produces them all.